# Bank Data Pipeline - Exploratory Data Analysis
Notebook ini menjalankan pemeriksaan row, kolom, schema, missing value, dan duplicate key sebelum ETL.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import functions as F


def find_project_root():
    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd() / 'ans_mentoring3')
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src/bank_etl').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not locate the ans_mentoring3 project root')


project_root = find_project_root()
sys.path.insert(0, str(project_root / 'src'))
load_dotenv(project_root / '.env')

if not Path('/.dockerenv').exists():
    local_overrides = {
        'SOURCE_DB_HOST': ('source_db', 'localhost'),
        'SOURCE_DB_PORT': ('5432', '5438'),
        'WAREHOUSE_DB_HOST': ('data_warehouse', 'localhost'),
        'WAREHOUSE_DB_PORT': ('5432', '5439'),
        'CSV_PATH': ('/app/data/new_bank_transactions.csv', str(project_root / 'data/new_bank_transactions.csv')),
        'JDBC_DRIVER_PATH': ('/opt/spark/jars/postgresql-42.6.0.jar', str(project_root / 'drivers/postgresql-42.6.0.jar')),
        'LOG_PATH': ('/app/logs/etl_pipeline.log', str(project_root / 'logs/etl_pipeline.log')),
        'PROFILE_REPORT_PATH': ('/app/logs/latest_profile_summary.json', str(project_root / 'logs/latest_profile_summary.json')),
    }
    for name, (container_default, local_default) in local_overrides.items():
        if os.getenv(name) in (None, container_default):
            os.environ[name] = local_default

from bank_etl.adapters.csv_reader import SparkCsvReader
from bank_etl.adapters.postgres_reader import PostgresJdbcReader
from bank_etl.domain.models import SourceTable
from bank_etl.infrastructure.config import Settings
from bank_etl.infrastructure.logger import configure_logger
from bank_etl.infrastructure.retry import RetryPolicy
from bank_etl.infrastructure.spark import SparkSessionFactory

settings = Settings.from_env()
configure_logger(settings.log_path)
spark = SparkSessionFactory(settings.spark).create()
retry = RetryPolicy(settings.retry)
database_reader = PostgresJdbcReader(spark, settings.source_db, retry)
csv_reader = SparkCsvReader(spark, retry)

2026-06-13 18:26:20.664 | INFO     | bank_etl.infrastructure.logger:configure_logger:57 | Centralized logging initialized at /app/logs/etl_pipeline.log


2026-06-13 18:26:20.665 | INFO     | bank_etl.infrastructure.spark:create:19 | Creating Spark session app_name=bank-etl-pipeline master=local[4] driver_memory=1g


26/06/13 18:26:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-13 18:26:24.048 | INFO     | bank_etl.infrastructure.spark:create:40 | Spark 3.5.5 initialized


In [2]:
def profile(frame, name, key=None):
    row_count = frame.count()
    print(f'{name}: {row_count:,} rows x {len(frame.columns)} columns')
    frame.printSchema()
    missing = frame.select([
        F.sum(F.when(F.col(column).isNull() | (F.trim(F.col(column).cast('string')) == ''), 1).otherwise(0)).alias(column)
        for column in frame.columns
    ])
    missing.show(truncate=False)
    if key:
        duplicate_count = frame.groupBy(key).count().filter(F.col('count') > 1).count()
        print(f'Duplicate {key}: {duplicate_count:,}')
    frame.show(5, truncate=False)

In [3]:
education = database_reader.read_table(SourceTable.EDUCATION_STATUS)
marital = database_reader.read_table(SourceTable.MARITAL_STATUS)
campaign = database_reader.read_table(SourceTable.MARKETING_CAMPAIGN)
transactions = csv_reader.read_csv(settings.csv_path)

2026-06-13 18:26:24.070 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:26 | Extracting PostgreSQL table education_status


2026-06-13 18:26:26.034 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:36 | Extracted 4 rows from education_status


2026-06-13 18:26:26.037 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:26 | Extracting PostgreSQL table marital_status


2026-06-13 18:26:26.247 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:36 | Extracted 3 rows from marital_status


2026-06-13 18:26:26.247 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:26 | Extracting PostgreSQL table marketing_campaign_deposit


2026-06-13 18:26:27.230 | INFO     | bank_etl.adapters.postgres_reader:_read_and_materialize:36 | Extracted 45211 rows from marketing_campaign_deposit


2026-06-13 18:26:27.233 | INFO     | bank_etl.adapters.csv_reader:_read_and_materialize:27 | Extracting CSV source /app/data/new_bank_transactions.csv


2026-06-13 18:26:31.734 | INFO     | bank_etl.adapters.csv_reader:_read_and_materialize:38 | Extracted 1048567 rows from CSV source


In [4]:
profile(education, 'education_status', 'education_id')
profile(marital, 'marital_status', 'marital_id')
profile(campaign, 'marketing_campaign_deposit', 'loan_data_id')

education_status: 4 rows x 4 columns
root
 |-- education_id: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



+------------+-----+----------+----------+
|education_id|value|created_at|updated_at|
+------------+-----+----------+----------+
|0           |0    |0         |0         |
+------------+-----+----------+----------+



Duplicate education_id: 0
+------------+---------+--------------------------+--------------------------+
|education_id|value    |created_at                |updated_at                |
+------------+---------+--------------------------+--------------------------+
|1           |tertiary |2025-02-28 15:31:04.358235|2025-02-28 15:31:04.358235|
|2           |secondary|2025-02-28 15:31:04.358235|2025-02-28 15:31:04.358235|
|3           |unknown  |2025-02-28 15:31:04.358235|2025-02-28 15:31:04.358235|
|4           |primary  |2025-02-28 15:31:04.358235|2025-02-28 15:31:04.358235|
+------------+---------+--------------------------+--------------------------+

marital_status: 3 rows x 4 columns
root
 |-- marital_id: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



+----------+-----+----------+----------+
|marital_id|value|created_at|updated_at|
+----------+-----+----------+----------+
|0         |0    |0         |0         |
+----------+-----+----------+----------+

Duplicate marital_id: 0
+----------+--------+--------------------------+--------------------------+
|marital_id|value   |created_at                |updated_at                |
+----------+--------+--------------------------+--------------------------+
|1         |married |2025-02-28 15:31:01.502136|2025-02-28 15:31:01.502136|
|2         |single  |2025-02-28 15:31:01.502136|2025-02-28 15:31:01.502136|
|3         |divorced|2025-02-28 15:31:01.502136|2025-02-28 15:31:01.502136|
+----------+--------+--------------------------+--------------------------+



marketing_campaign_deposit: 45,211 rows x 20 columns
root
 |-- loan_data_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital_id: integer (nullable = true)
 |-- education_id: integer (nullable = true)
 |-- default: boolean (nullable = true)
 |-- balance: string (nullable = true)
 |-- housing: boolean (nullable = true)
 |-- loan: boolean (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- subscribed_deposit: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



+------------+---+---+----------+------------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+------------------+----------+----------+
|loan_data_id|age|job|marital_id|education_id|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|subscribed_deposit|created_at|updated_at|
+------------+---+---+----------+------------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+------------------+----------+----------+
|0           |0  |0  |0         |0           |0      |0      |0      |0   |0      |0  |0    |0       |0       |0    |0       |0       |0                 |0         |0         |
+------------+---+---+----------+------------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+------------------+----------+----------+

Duplicate loan_data_id: 0


+------------+---+------------+----------+------------+-------+-------+-------+-----+-------+---+-----+--------+--------+-----+--------+--------+------------------+--------------------------+--------------------------+
|loan_data_id|age|job         |marital_id|education_id|default|balance|housing|loan |contact|day|month|duration|campaign|pdays|previous|poutcome|subscribed_deposit|created_at                |updated_at                |
+------------+---+------------+----------+------------+-------+-------+-------+-----+-------+---+-----+--------+--------+-----+--------+--------+------------------+--------------------------+--------------------------+
|1           |58 |management  |1         |1           |false  |$2143  |true   |false|unknown|5  |may  |261     |1       |-1   |0       |unknown |false             |2025-02-28 15:59:11.102813|2025-02-28 15:59:11.102813|
|2           |44 |technician  |2         |2           |false  |$29    |true   |false|unknown|5  |may  |151     |1       |-1 

In [5]:
profile(transactions, 'new_bank_transactions.csv', 'TransactionID')

new_bank_transactions.csv: 1,048,567 rows x 9 columns
root
 |-- TransactionID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerDOB: string (nullable = true)
 |-- CustGender: string (nullable = true)
 |-- CustLocation: string (nullable = true)
 |-- CustAccountBalance: string (nullable = true)
 |-- TransactionDate: string (nullable = true)
 |-- TransactionTime: string (nullable = true)
 |-- TransactionAmount (INR): string (nullable = true)



+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|TransactionID|CustomerID|CustomerDOB|CustGender|CustLocation|CustAccountBalance|TransactionDate|TransactionTime|TransactionAmount (INR)|
+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|0            |0         |0          |1100      |151         |2369              |0              |0              |0                      |
+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+



Duplicate TransactionID: 0
+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|TransactionID|CustomerID|CustomerDOB|CustGender|CustLocation|CustAccountBalance|TransactionDate|TransactionTime|TransactionAmount (INR)|
+-------------+----------+-----------+----------+------------+------------------+---------------+---------------+-----------------------+
|T642232      |C1010028  |25/8/88    |F         |DELHI       |296828.37         |29/8/16        |95212          |557                    |
|T87414       |C1010035  |2/3/92     |M         |MUMBAI      |7284.42           |1/8/16         |111917         |50                     |
|T560676      |C1010035_2|9/6/80     |M         |NAVI MUMBAI |378013.09         |27/8/16        |185011         |700                    |
|T610204      |C1010036  |26/2/96    |M         |GURGAON     |355430.17         |26/8/16        |95203          |208                    |
|T95766

In [6]:
for frame in (education, marital, campaign, transactions):
    frame.unpersist(blocking=False)
spark.stop()